# C / C++ 指標運算 50 題：自動檢查版

本題本依照你提供的 C / C++ 自動檢查 Notebook 版型製作，主題集中在 **指標運算**。

## 作答流程

1. 先執行下面的「C / C++ 自動檢查工具」cell。
2. 每題都有兩個作答 cell：
   - `%%c_answer Q...`：C（C11）
   - `%%cpp_answer Q...`：C++（C++17）
3. **不需要在作答 cell 手動輸入測試資料。**
4. 完成某一語言後，執行緊接在該作答 cell 下方的檢查。
5. 每題使用 **3 組測試資料**。
6. 最後執行「總檢查」查看完成度。

> 可以只練 C、只練 C++，或同一題兩種都寫。題目由基礎逐步增加整合程度。


In [ ]:
# === C / C++ 自動檢查工具：請先執行這個 cell，不要修改 ===
import base64
import json
import os
import shutil
import subprocess
import tempfile
from IPython.core.magic import register_cell_magic

_TEST_CASES = json.loads(
    base64.b64decode("eyJRMSI6IFt7ImlucHV0IjogIjNcbjEgMiAzXG4iLCAiZXhwZWN0ZWQiOiAiMSAyIDNcbiJ9LCB7ImlucHV0IjogIjJcbi0xIDVcbiIsICJleHBlY3RlZCI6ICItMSA1XG4ifSwgeyJpbnB1dCI6ICIxXG45XG4iLCAiZXhwZWN0ZWQiOiAiOVxuIn1dLCAiUTIiOiBbeyJpbnB1dCI6ICI0XG4xIDIgMyA0XG4iLCAiZXhwZWN0ZWQiOiAiMTBcbiJ9LCB7ImlucHV0IjogIjNcbi0yIDUgMVxuIiwgImV4cGVjdGVkIjogIjRcbiJ9LCB7ImlucHV0IjogIjFcbjdcbiIsICJleHBlY3RlZCI6ICI3XG4ifV0sICJRMyI6IFt7ImlucHV0IjogIjRcbjEgMiAzIDRcbiIsICJleHBlY3RlZCI6ICI0IDMgMiAxXG4ifSwgeyJpbnB1dCI6ICIyXG4tMSA5XG4iLCAiZXhwZWN0ZWQiOiAiOSAtMVxuIn0sIHsiaW5wdXQiOiAiMVxuNVxuIiwgImV4cGVjdGVkIjogIjVcbiJ9XSwgIlE0IjogW3siaW5wdXQiOiAiNVxuMSAyIDMgNCA1XG4iLCAiZXhwZWN0ZWQiOiAiOVxuIn0sIHsiaW5wdXQiOiAiNFxuMTAgMjAgMzAgNDBcbiIsICJleHBlY3RlZCI6ICI0MFxuIn0sIHsiaW5wdXQiOiAiMVxuN1xuIiwgImV4cGVjdGVkIjogIjdcbiJ9XSwgIlE1IjogW3siaW5wdXQiOiAiNVxuMSAyIDMgNCA1XG4iLCAiZXhwZWN0ZWQiOiAiNlxuIn0sIHsiaW5wdXQiOiAiNFxuMTAgMjAgMzAgNDBcbiIsICJleHBlY3RlZCI6ICI2MFxuIn0sIHsiaW5wdXQiOiAiMVxuN1xuIiwgImV4cGVjdGVkIjogIjBcbiJ9XSwgIlE2IjogW3siaW5wdXQiOiAiNFxuMyA2IDkgMTJcbjJcbiIsICJleHBlY3RlZCI6ICI5XG4ifSwgeyJpbnB1dCI6ICIzXG4tMSA4IDVcbjBcbiIsICJleHBlY3RlZCI6ICItMVxuIn0sIHsiaW5wdXQiOiAiMlxuNyA0XG4xXG4iLCAiZXhwZWN0ZWQiOiAiNFxuIn1dLCAiUTciOiBbeyJpbnB1dCI6ICI1XG4xIDIgMyA0IDVcbjJcbiIsICJleHBlY3RlZCI6ICIxMlxuIn0sIHsiaW5wdXQiOiAiM1xuNSA2IDdcbjBcbiIsICJleHBlY3RlZCI6ICIxOFxuIn0sIHsiaW5wdXQiOiAiNFxuMSAxIDEgMVxuM1xuIiwgImV4cGVjdGVkIjogIjFcbiJ9XSwgIlE4IjogW3siaW5wdXQiOiAiNVxuMSAyIDMgNCA1XG4zXG4iLCAiZXhwZWN0ZWQiOiAiNlxuIn0sIHsiaW5wdXQiOiAiM1xuNSA2IDdcbjBcbiIsICJleHBlY3RlZCI6ICIwXG4ifSwgeyJpbnB1dCI6ICI0XG4yIDIgMiAyXG40XG4iLCAiZXhwZWN0ZWQiOiAiOFxuIn1dLCAiUTkiOiBbeyJpbnB1dCI6ICI0XG4zIDkgMiA3XG4iLCAiZXhwZWN0ZWQiOiAiOVxuIn0sIHsiaW5wdXQiOiAiM1xuLTggLTIgLTVcbiIsICJleHBlY3RlZCI6ICItMlxuIn0sIHsiaW5wdXQiOiAiMVxuNFxuIiwgImV4cGVjdGVkIjogIjRcbiJ9XSwgIlExMCI6IFt7ImlucHV0IjogIjRcbjMgOSAyIDdcbiIsICJleHBlY3RlZCI6ICIyXG4ifSwgeyJpbnB1dCI6ICIzXG4tOCAtMiAtNVxuIiwgImV4cGVjdGVkIjogIi04XG4ifSwgeyJpbnB1dCI6ICIxXG40XG4iLCAiZXhwZWN0ZWQiOiAiNFxuIn1dLCAiUTExIjogW3siaW5wdXQiOiAiM1xuMSAyIDNcbiIsICJleHBlY3RlZCI6ICIyIDMgNFxuIn0sIHsiaW5wdXQiOiAiMlxuLTEgMFxuIiwgImV4cGVjdGVkIjogIjAgMVxuIn0sIHsiaW5wdXQiOiAiMVxuOVxuIiwgImV4cGVjdGVkIjogIjEwXG4ifV0sICJRMTIiOiBbeyJpbnB1dCI6ICIzXG4xIDIgM1xuIiwgImV4cGVjdGVkIjogIjAgMSAyXG4ifSwgeyJpbnB1dCI6ICIyXG4wIC0yXG4iLCAiZXhwZWN0ZWQiOiAiLTEgLTNcbiJ9LCB7ImlucHV0IjogIjFcbjlcbiIsICJleHBlY3RlZCI6ICI4XG4ifV0sICJRMTMiOiBbeyJpbnB1dCI6ICI0XG4xIDIgMyA0XG4iLCAiZXhwZWN0ZWQiOiAiMTBcbiJ9LCB7ImlucHV0IjogIjNcbi0xIDIgOVxuIiwgImV4cGVjdGVkIjogIjEwXG4ifSwgeyJpbnB1dCI6ICIxXG4wXG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn1dLCAiUTE0IjogW3siaW5wdXQiOiAiNVxuMSAyIDMgNCA1XG4xIDRcbiIsICJleHBlY3RlZCI6ICIzXG4ifSwgeyJpbnB1dCI6ICI0XG44IDcgNiA1XG4zIDBcbiIsICJleHBlY3RlZCI6ICItM1xuIn0sIHsiaW5wdXQiOiAiMlxuOSA5XG4wIDBcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJRMTUiOiBbeyJpbnB1dCI6ICI1XG40IDcgMiA3IDlcbjdcbiIsICJleHBlY3RlZCI6ICIxXG4ifSwgeyJpbnB1dCI6ICIzXG4xIDIgM1xuOFxuIiwgImV4cGVjdGVkIjogIi0xXG4ifSwgeyJpbnB1dCI6ICIxXG41XG41XG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn1dLCAiUTE2IjogW3siaW5wdXQiOiAiNVxuNCA3IDIgNyA5XG43XG4iLCAiZXhwZWN0ZWQiOiAiM1xuIn0sIHsiaW5wdXQiOiAiM1xuMSAyIDNcbjhcbiIsICJleHBlY3RlZCI6ICItMVxuIn0sIHsiaW5wdXQiOiAiM1xuNSA1IDVcbjVcbiIsICJleHBlY3RlZCI6ICIyXG4ifV0sICJRMTciOiBbeyJpbnB1dCI6ICI1XG4zIDkgMiA3IDFcbiIsICJleHBlY3RlZCI6ICIzXG4ifSwgeyJpbnB1dCI6ICI0XG44IDIgNSA5XG4iLCAiZXhwZWN0ZWQiOiAiMlxuIn0sIHsiaW5wdXQiOiAiM1xuNCA0IDRcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJRMTgiOiBbeyJpbnB1dCI6ICI0XG4xIDIgMyA0XG4iLCAiZXhwZWN0ZWQiOiAiNFxuIn0sIHsiaW5wdXQiOiAiMVxuOVxuIiwgImV4cGVjdGVkIjogIjFcbiJ9LCB7ImlucHV0IjogIjZcbjAgMCAwIDAgMCAwXG4iLCAiZXhwZWN0ZWQiOiAiNlxuIn1dLCAiUTE5IjogW3siaW5wdXQiOiAiM1xuMiA0IDZcbiIsICJleHBlY3RlZCI6ICIyIDQgNlxuIn0sIHsiaW5wdXQiOiAiMlxuLTEgOFxuIiwgImV4cGVjdGVkIjogIi0xIDhcbiJ9LCB7ImlucHV0IjogIjFcbjBcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJRMjAiOiBbeyJpbnB1dCI6ICIzXG4xIDIgM1xuNCA1IDZcbiIsICJleHBlY3RlZCI6ICI1IDcgOVxuIn0sIHsiaW5wdXQiOiAiMlxuLTEgMlxuMyAtNFxuIiwgImV4cGVjdGVkIjogIjIgLTJcbiJ9LCB7ImlucHV0IjogIjFcbjdcbjhcbiIsICJleHBlY3RlZCI6ICIxNVxuIn1dLCAiUTIxIjogW3siaW5wdXQiOiAiaGVsbG9cbiIsICJleHBlY3RlZCI6ICI1XG4ifSwgeyJpbnB1dCI6ICJhXG4iLCAiZXhwZWN0ZWQiOiAiMVxuIn0sIHsiaW5wdXQiOiAicG9pbnRlclxuIiwgImV4cGVjdGVkIjogIjdcbiJ9XSwgIlEyMiI6IFt7ImlucHV0IjogImhlbGxvXG4iLCAiZXhwZWN0ZWQiOiAib2xsZWhcbiJ9LCB7ImlucHV0IjogImFiY1xuIiwgImV4cGVjdGVkIjogImNiYVxuIn0sIHsiaW5wdXQiOiAieFxuIiwgImV4cGVjdGVkIjogInhcbiJ9XSwgIlEyMyI6IFt7ImlucHV0IjogImVkdWNhdGlvblxuIiwgImV4cGVjdGVkIjogIjVcbiJ9LCB7ImlucHV0IjogIlhZWlxuIiwgImV4cGVjdGVkIjogIjBcbiJ9LCB7ImlucHV0IjogIkFwcGxlXG4iLCAiZXhwZWN0ZWQiOiAiMlxuIn1dLCAiUTI0IjogW3siaW5wdXQiOiAiYWJjZFxuIiwgImV4cGVjdGVkIjogImRjYmFcbiJ9LCB7ImlucHV0IjogImxldmVsXG4iLCAiZXhwZWN0ZWQiOiAibGV2ZWxcbiJ9LCB7ImlucHV0IjogImFcbiIsICJleHBlY3RlZCI6ICJhXG4ifV0sICJRMjUiOiBbeyJpbnB1dCI6ICJyYWNlY2FyXG4iLCAiZXhwZWN0ZWQiOiAiMVxuIn0sIHsiaW5wdXQiOiAicG9pbnRlclxuIiwgImV4cGVjdGVkIjogIjBcbiJ9LCB7ImlucHV0IjogImFhXG4iLCAiZXhwZWN0ZWQiOiAiMVxuIn1dLCAiUTI2IjogW3siaW5wdXQiOiAiYTFiMjNcbiIsICJleHBlY3RlZCI6ICIzXG4ifSwgeyJpbnB1dCI6ICIwMDdcbiIsICJleHBlY3RlZCI6ICIzXG4ifSwgeyJpbnB1dCI6ICJhYmNcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJRMjciOiBbeyJpbnB1dCI6ICJhMWIyM1xuIiwgImV4cGVjdGVkIjogIjZcbiJ9LCB7ImlucHV0IjogIjAwN1xuIiwgImV4cGVjdGVkIjogIjdcbiJ9LCB7ImlucHV0IjogImFiY1xuIiwgImV4cGVjdGVkIjogIjBcbiJ9XSwgIlEyOCI6IFt7ImlucHV0IjogImJhbmFuYSBuXG4iLCAiZXhwZWN0ZWQiOiAiMlxuIn0sIHsiaW5wdXQiOiAiYXBwbGUgYVxuIiwgImV4cGVjdGVkIjogIjBcbiJ9LCB7ImlucHV0IjogImNhdCB6XG4iLCAiZXhwZWN0ZWQiOiAiLTFcbiJ9XSwgIlEyOSI6IFt7ImlucHV0IjogImJhbmFuYSBhXG4iLCAiZXhwZWN0ZWQiOiAiNVxuIn0sIHsiaW5wdXQiOiAiaGVsbG8gbFxuIiwgImV4cGVjdGVkIjogIjNcbiJ9LCB7ImlucHV0IjogImNhdCB6XG4iLCAiZXhwZWN0ZWQiOiAiLTFcbiJ9XSwgIlEzMCI6IFt7ImlucHV0IjogImJhbmFuYSBhIG9cbiIsICJleHBlY3RlZCI6ICJib25vbm9cbiJ9LCB7ImlucHV0IjogImhlbGxvIGwgeFxuIiwgImV4cGVjdGVkIjogImhleHhvXG4ifSwgeyJpbnB1dCI6ICJhYmMgeiBxXG4iLCAiZXhwZWN0ZWQiOiAiYWJjXG4ifV0sICJRMzEiOiBbeyJpbnB1dCI6ICIzXG4xLjUgMi4wIDMuNVxuIiwgImV4cGVjdGVkIjogIjcuMDBcbiJ9LCB7ImlucHV0IjogIjJcbi0xLjI1IDAuMjVcbiIsICJleHBlY3RlZCI6ICItMS4wMFxuIn0sIHsiaW5wdXQiOiAiMVxuNC4yXG4iLCAiZXhwZWN0ZWQiOiAiNC4yMFxuIn1dLCAiUTMyIjogW3siaW5wdXQiOiAiM1xuMiA0IDZcbiIsICJleHBlY3RlZCI6ICIzLjAwIDYuMDAgOS4wMFxuIn0sIHsiaW5wdXQiOiAiMlxuLTIgMVxuIiwgImV4cGVjdGVkIjogIi0zLjAwIDEuNTBcbiJ9LCB7ImlucHV0IjogIjFcbjBcbiIsICJleHBlY3RlZCI6ICIwLjAwXG4ifV0sICJRMzMiOiBbeyJpbnB1dCI6ICIzXG4xIDIgNlxuIiwgImV4cGVjdGVkIjogIjMuMDBcbiJ9LCB7ImlucHV0IjogIjJcbjEuNSAyLjVcbiIsICJleHBlY3RlZCI6ICIyLjAwXG4ifSwgeyJpbnB1dCI6ICIxXG4tNFxuIiwgImV4cGVjdGVkIjogIi00LjAwXG4ifV0sICJRMzQiOiBbeyJpbnB1dCI6ICI0XG41IDEgOSAzXG4wIDJcbiIsICJleHBlY3RlZCI6ICJSSUdIVFxuIn0sIHsiaW5wdXQiOiAiM1xuNyAyIDFcbjAgMlxuIiwgImV4cGVjdGVkIjogIkxFRlRcbiJ9LCB7ImlucHV0IjogIjJcbjQgNFxuMCAxXG4iLCAiZXhwZWN0ZWQiOiAiRVFVQUxcbiJ9XSwgIlEzNSI6IFt7ImlucHV0IjogIjVcbjEgMiAzIDQgNVxuMSAzXG4iLCAiZXhwZWN0ZWQiOiAiOVxuIn0sIHsiaW5wdXQiOiAiNFxuOCAtMiA1IDFcbjAgMFxuIiwgImV4cGVjdGVkIjogIjhcbiJ9LCB7ImlucHV0IjogIjNcbjIgMiAyXG4wIDJcbiIsICJleHBlY3RlZCI6ICI2XG4ifV0sICJRMzYiOiBbeyJpbnB1dCI6ICI1XG4wIC0yIDAgNyA5XG4iLCAiZXhwZWN0ZWQiOiAiMlxuIn0sIHsiaW5wdXQiOiAiM1xuLTEgLTIgLTNcbiIsICJleHBlY3RlZCI6ICItMVxuIn0sIHsiaW5wdXQiOiAiNFxuNSAwIC0xIDJcbiIsICJleHBlY3RlZCI6ICIyXG4ifV0sICJRMzciOiBbeyJpbnB1dCI6ICI1XG4xIDIgNCA3IDExXG45XG4iLCAiZXhwZWN0ZWQiOiAiMVxuIn0sIHsiaW5wdXQiOiAiNFxuMSAzIDUgOFxuMjBcbiIsICJleHBlY3RlZCI6ICIwXG4ifSwgeyJpbnB1dCI6ICIyXG4yIDJcbjRcbiIsICJleHBlY3RlZCI6ICIxXG4ifV0sICJRMzgiOiBbeyJpbnB1dCI6ICI1XG4xIDIgMyA0IDVcbiIsICJleHBlY3RlZCI6ICI1IDQgMyAyIDFcbiJ9LCB7ImlucHV0IjogIjRcbi0xIDAgMiA4XG4iLCAiZXhwZWN0ZWQiOiAiOCAyIDAgLTFcbiJ9LCB7ImlucHV0IjogIjFcbjdcbiIsICJleHBlY3RlZCI6ICI3XG4ifV0sICJRMzkiOiBbeyJpbnB1dCI6ICI0XG4xIDIgMyA0XG4iLCAiZXhwZWN0ZWQiOiAiNCAxIDIgM1xuIn0sIHsiaW5wdXQiOiAiMVxuNVxuIiwgImV4cGVjdGVkIjogIjVcbiJ9LCB7ImlucHV0IjogIjNcbi0xIDAgOVxuIiwgImV4cGVjdGVkIjogIjkgLTEgMFxuIn1dLCAiUTQwIjogW3siaW5wdXQiOiAiNVxuMSAyIDMgNCA1XG4iLCAiZXhwZWN0ZWQiOiAiNiA2IDZcbiJ9LCB7ImlucHV0IjogIjRcbjEgMiAzIDRcbiIsICJleHBlY3RlZCI6ICI1IDVcbiJ9LCB7ImlucHV0IjogIjFcbjdcbiIsICJleHBlY3RlZCI6ICIxNFxuIn1dLCAiUTQxIjogW3siaW5wdXQiOiAiNFxuMSAyIDMgNFxuIiwgImV4cGVjdGVkIjogIjEwXG4ifSwgeyJpbnB1dCI6ICIzXG4tNSAxIDJcbiIsICJleHBlY3RlZCI6ICItMlxuIn0sIHsiaW5wdXQiOiAiMVxuOFxuIiwgImV4cGVjdGVkIjogIjhcbiJ9XSwgIlE0MiI6IFt7ImlucHV0IjogIjVcbjEgLTIgMyAwIDRcbiIsICJleHBlY3RlZCI6ICIzXG4ifSwgeyJpbnB1dCI6ICIzXG4tMSAwIC0yXG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn0sIHsiaW5wdXQiOiAiMlxuNiA3XG4iLCAiZXhwZWN0ZWQiOiAiMlxuIn1dLCAiUTQzIjogW3siaW5wdXQiOiAiM1xuMSA5IDRcbiIsICJleHBlY3RlZCI6ICIxIDkgNFxuIn0sIHsiaW5wdXQiOiAiMlxuLTMgOFxuIiwgImV4cGVjdGVkIjogIi0zIDhcbiJ9LCB7ImlucHV0IjogIjFcbjBcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJRNDQiOiBbeyJpbnB1dCI6ICIzXG4xIDIgM1xuMSAyIDNcbiIsICJleHBlY3RlZCI6ICIxXG4ifSwgeyJpbnB1dCI6ICIzXG4xIDIgM1xuMSAyIDRcbiIsICJleHBlY3RlZCI6ICIwXG4ifSwgeyJpbnB1dCI6ICIxXG41XG41XG4iLCAiZXhwZWN0ZWQiOiAiMVxuIn1dLCAiUTQ1IjogW3siaW5wdXQiOiAiM1xuMSAyIDNcbjEgNCAwXG4iLCAiZXhwZWN0ZWQiOiAiLTFcbiJ9LCB7ImlucHV0IjogIjJcbjUgMVxuNCA5XG4iLCAiZXhwZWN0ZWQiOiAiMVxuIn0sIHsiaW5wdXQiOiAiM1xuMSAyIDNcbjEgMiAzXG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn1dLCAiUTQ2IjogW3siaW5wdXQiOiAiNlxuMSAwIDIgMCAwIDNcbiIsICJleHBlY3RlZCI6ICIzXG4xIDIgM1xuIn0sIHsiaW5wdXQiOiAiM1xuMCAwIDBcbiIsICJleHBlY3RlZCI6ICIwXG5FTVBUWVxuIn0sIHsiaW5wdXQiOiAiM1xuNCA1IDZcbiIsICJleHBlY3RlZCI6ICIzXG40IDUgNlxuIn1dLCAiUTQ3IjogW3siaW5wdXQiOiAiN1xuMSAxIDIgMiAyIDUgNVxuIiwgImV4cGVjdGVkIjogIjNcbjEgMiA1XG4ifSwgeyJpbnB1dCI6ICI0XG4zIDMgMyAzXG4iLCAiZXhwZWN0ZWQiOiAiMVxuM1xuIn0sIHsiaW5wdXQiOiAiM1xuMSAyIDNcbiIsICJleHBlY3RlZCI6ICIzXG4xIDIgM1xuIn1dLCAiUTQ4IjogW3siaW5wdXQiOiAiMyAzXG4xIDQgN1xuMiAzIDhcbiIsICJleHBlY3RlZCI6ICIxIDIgMyA0IDcgOFxuIn0sIHsiaW5wdXQiOiAiMiAxXG4tMiA1XG4wXG4iLCAiZXhwZWN0ZWQiOiAiLTIgMCA1XG4ifSwgeyJpbnB1dCI6ICIxIDJcbjNcbjMgM1xuIiwgImV4cGVjdGVkIjogIjMgMyAzXG4ifV0sICJRNDkiOiBbeyJpbnB1dCI6ICI0IDVcbjEgMiA0IDdcbjIgMyA0IDQgOFxuIiwgImV4cGVjdGVkIjogIjJcbiJ9LCB7ImlucHV0IjogIjMgM1xuMSAxIDJcbjEgMSAxXG4iLCAiZXhwZWN0ZWQiOiAiMlxuIn0sIHsiaW5wdXQiOiAiMiAyXG41IDZcbjEgMlxuIiwgImV4cGVjdGVkIjogIjBcbiJ9XSwgIlE1MCI6IFt7ImlucHV0IjogIjVcbjEgMiA0IDcgMTFcbjlcbiIsICJleHBlY3RlZCI6ICIxIDNcbiJ9LCB7ImlucHV0IjogIjRcbjEgMyA1IDhcbjIwXG4iLCAiZXhwZWN0ZWQiOiAiLTEgLTFcbiJ9LCB7ImlucHV0IjogIjRcbjEgMiAyIDNcbjRcbiIsICJleHBlY3RlZCI6ICIwIDNcbiJ9XX0=").decode("utf-8")
)

_ANSWERS = {"C": {}, "CPP": {}}
_RESULTS = {"C": {}, "CPP": {}}

def _normalise_output(text):
    lines = [line.rstrip() for line in text.replace("\r\n", "\n").split("\n")]
    while lines and lines[-1] == "":
        lines.pop()
    return "\n".join(lines)

def _compile_and_run(source, input_text, language):
    if language == "C":
        compiler = shutil.which("gcc")
        suffix = ".c"
        flags = ["-std=c11", "-O2"]
    else:
        compiler = shutil.which("g++")
        suffix = ".cpp"
        flags = ["-std=c++17", "-O2"]

    if compiler is None:
        return False, "", f"找不到 {'gcc' if language == 'C' else 'g++'}。請先安裝 C/C++ compiler。"

    with tempfile.TemporaryDirectory() as td:
        src = os.path.join(td, "answer" + suffix)
        exe = os.path.join(td, "answer")
        with open(src, "w", encoding="utf-8") as f:
            f.write(source)

        compile_result = subprocess.run(
            [compiler, src, "-o", exe] + flags,
            capture_output=True,
            text=True,
        )
        if compile_result.returncode != 0:
            return False, "", compile_result.stderr

        try:
            run_result = subprocess.run(
                [exe],
                input=input_text,
                capture_output=True,
                text=True,
                timeout=5,
            )
        except subprocess.TimeoutExpired:
            return False, "", "程式執行超過 5 秒，請檢查是否有無限迴圈。"

        if run_result.returncode != 0:
            err = run_result.stderr or f"程式結束碼：{run_result.returncode}"
            return False, run_result.stdout, err
        return True, run_result.stdout, ""

@register_cell_magic
def c_answer(line, cell):
    qid = line.strip().upper()
    _ANSWERS["C"][qid] = cell
    print(f"已儲存 {qid} 的 C 作答。請執行下方 C 檢查 cell。")

@register_cell_magic
def cpp_answer(line, cell):
    qid = line.strip().upper()
    _ANSWERS["CPP"][qid] = cell
    print(f"已儲存 {qid} 的 C++ 作答。請執行下方 C++ 檢查 cell。")

def _run_check(qid, language):
    qid = qid.upper()
    language = language.upper()
    label = "C" if language == "C" else "C++"

    if qid not in _ANSWERS[language]:
        print(f"❌ {qid}：尚未執行 {label} 作答 cell")
        _RESULTS[language][qid] = False
        return

    source = _ANSWERS[language][qid]
    cases = _TEST_CASES[qid]
    passed = 0

    for case_number, case in enumerate(cases, start=1):
        ok_run, stdout, error = _compile_and_run(source, case["input"], language)
        if not ok_run:
            print(f"❌ {qid}（{label}）：編譯或執行失敗")
            print(error)
            _RESULTS[language][qid] = False
            return

        actual = _normalise_output(stdout)
        expected = _normalise_output(case["expected"])

        if actual == expected:
            passed += 1
        else:
            print(f"❌ {qid}（{label}）測試 {case_number} 未通過")
            print("測試輸入：")
            print(case["input"], end="")
            print("你的輸出：")
            print(stdout if stdout else "<沒有輸出>")
            print("預期輸出：")
            print(case["expected"], end="")
            print()

    success = passed == len(cases)
    _RESULTS[language][qid] = success
    if success:
        print(f"✅ {qid}（{label}）全部通過（{passed} / {len(cases)}）")
    else:
        print(f"❌ {qid}（{label}）：通過 {passed} / {len(cases)} 個測試")

def _language_report(language):
    label = "C" if language == "C" else "C++"
    results = _RESULTS[language]
    passed = sum(results.get(f"Q{i}", False) for i in range(1, 51))
    untouched = [f"Q{i}" for i in range(1, 51) if f"Q{i}" not in results]
    print(f"{label}：{passed} / 50 題通過")
    if untouched:
        print("尚未執行檢查：" + ", ".join(untouched))

def _final_report():
    print("=== C / C++ 總檢查 ===")
    _language_report("C")
    print()
    _language_report("CPP")
    both_passed = sum(
        _RESULTS["C"].get(f"Q{i}", False)
        and _RESULTS["CPP"].get(f"Q{i}", False)
        for i in range(1, 51)
    )
    print()
    print(f"C 與 C++ 都通過：{both_passed} / 50 題")

print("C / C++ 自動檢查工具已載入。")
print("C 使用 C11；C++ 使用 C++17。")
print("每題有 3 組測試資料；作答後不需要手動輸入測試資料。")


## 作答方式提醒

每題流程：

```text
題目
↓
C 作答 cell ─────→ C 檢查
↓
C++ 作答 cell ───→ C++ 檢查
↓
同一組 3 組測試資料
↓
✅ 通過 / ❌ 顯示差異
```

只修改 `%%c_answer Q...` 與 `%%cpp_answer Q...` cell 裡的程式內容；不要修改檢查 cell。

**重要：** 自動檢查主要驗證輸出正確性；題目要求使用 pointer / array / function pointer 的部分，請自行遵守，才能真正練到該章觀念。


---

# Part A：p+i、p++、p--（Q1–Q10）


## Q1. 使用 *(p+i) 輸出

讀入 n 個整數，令 p=a，使用 `*(p+i)` 依序輸出。

**練習重點：** `p+i` + dereference

**範例輸入：**

```text
3
1 2 3
```

**範例輸出：**

```text
1 2 3
```


### C 作答

In [ ]:
%%c_answer Q1
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q1

In [ ]:
# 檢查 Q1（C）：完成上方 C 程式後執行
_run_check("Q1", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q1
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q1

In [ ]:
# 檢查 Q1（C++）：完成上方 C++ 程式後執行
_run_check("Q1", "CPP")


## Q2. 用 p++ 求和

讀入陣列，以 `p++` 方式走訪並求和。

**練習重點：** post-increment pointer

**範例輸入：**

```text
4
1 2 3 4
```

**範例輸出：**

```text
10
```


### C 作答

In [ ]:
%%c_answer Q2
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q2

In [ ]:
# 檢查 Q2（C）：完成上方 C 程式後執行
_run_check("Q2", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q2
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q2

In [ ]:
# 檢查 Q2（C++）：完成上方 C++ 程式後執行
_run_check("Q2", "CPP")


## Q3. 從尾端用 p-- 輸出

讀入陣列，令 p 指向最後元素，使用 `p--` 反向輸出。

**練習重點：** pointer decrement

**範例輸入：**

```text
4
1 2 3 4
```

**範例輸出：**

```text
4 3 2 1
```


### C 作答

In [ ]:
%%c_answer Q3
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q3

In [ ]:
# 檢查 Q3（C）：完成上方 C 程式後執行
_run_check("Q3", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q3
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q3

In [ ]:
# 檢查 Q3（C++）：完成上方 C++ 程式後執行
_run_check("Q3", "CPP")


## Q4. 每隔一格求和

使用 `p += 2` 計算索引 0,2,4,... 的元素總和。

**練習重點：** pointer step

**範例輸入：**

```text
5
1 2 3 4 5
```

**範例輸出：**

```text
9
```


### C 作答

In [ ]:
%%c_answer Q4
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q4

In [ ]:
# 檢查 Q4（C）：完成上方 C 程式後執行
_run_check("Q4", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q4
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q4

In [ ]:
# 檢查 Q4（C++）：完成上方 C++ 程式後執行
_run_check("Q4", "CPP")


## Q5. 奇數索引求和

計算索引 1,3,5,... 的元素總和，練習從 `a+1` 開始每次加 2。

**練習重點：** pointer offset + step

**範例輸入：**

```text
5
1 2 3 4 5
```

**範例輸出：**

```text
6
```


### C 作答

In [ ]:
%%c_answer Q5
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q5

In [ ]:
# 檢查 Q5（C）：完成上方 C 程式後執行
_run_check("Q5", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q5
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q5

In [ ]:
# 檢查 Q5（C++）：完成上方 C++ 程式後執行
_run_check("Q5", "CPP")


## Q6. 第 k 個偏移值

讀入 n、陣列、k（0<=k<n），用 `*(a+k)` 輸出元素。

**練習重點：** pointer offset

**範例輸入：**

```text
4
3 6 9 12
2
```

**範例輸出：**

```text
9
```


### C 作答

In [ ]:
%%c_answer Q6
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q6

In [ ]:
# 檢查 Q6（C）：完成上方 C 程式後執行
_run_check("Q6", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q6
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q6

In [ ]:
# 檢查 Q6（C++）：完成上方 C++ 程式後執行
_run_check("Q6", "CPP")


## Q7. 從 k 開始的總和

讀入 k，令 p=a+k，使用 pointer 求 k 到尾端總和。

**練習重點：** offset pointer traversal

**範例輸入：**

```text
5
1 2 3 4 5
2
```

**範例輸出：**

```text
12
```


### C 作答

In [ ]:
%%c_answer Q7
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q7

In [ ]:
# 檢查 Q7（C）：完成上方 C 程式後執行
_run_check("Q7", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q7
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q7

In [ ]:
# 檢查 Q7（C++）：完成上方 C++ 程式後執行
_run_check("Q7", "CPP")


## Q8. 前 k 個總和

使用 begin pointer 與 `a+k` 作為 end，求前 k 個元素和。

**練習重點：** half-open pointer range

**範例輸入：**

```text
5
1 2 3 4 5
3
```

**範例輸出：**

```text
6
```


### C 作答

In [ ]:
%%c_answer Q8
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q8

In [ ]:
# 檢查 Q8（C）：完成上方 C 程式後執行
_run_check("Q8", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q8
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q8

In [ ]:
# 檢查 Q8（C++）：完成上方 C++ 程式後執行
_run_check("Q8", "CPP")


## Q9. 用 pointer 找最大值

透過 pointer increment 掃描陣列，輸出最大值。

**練習重點：** pointer traversal

**範例輸入：**

```text
4
3 9 2 7
```

**範例輸出：**

```text
9
```


### C 作答

In [ ]:
%%c_answer Q9
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q9

In [ ]:
# 檢查 Q9（C）：完成上方 C 程式後執行
_run_check("Q9", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q9
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q9

In [ ]:
# 檢查 Q9（C++）：完成上方 C++ 程式後執行
_run_check("Q9", "CPP")


## Q10. 用 pointer 找最小值

透過 pointer increment 掃描陣列，輸出最小值。

**練習重點：** pointer traversal

**範例輸入：**

```text
4
3 9 2 7
```

**範例輸出：**

```text
2
```


### C 作答

In [ ]:
%%c_answer Q10
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q10

In [ ]:
# 檢查 Q10（C）：完成上方 C 程式後執行
_run_check("Q10", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q10
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q10

In [ ]:
# 檢查 Q10（C++）：完成上方 C++ 程式後執行
_run_check("Q10", "CPP")


---

# Part B：begin/end 與 pointer distance（Q11–Q20）


## Q11. p++ 原地加 1

用 moving pointer 將所有元素加 1，輸出陣列。

**練習重點：** pointer increment + write

**範例輸入：**

```text
3
1 2 3
```

**範例輸出：**

```text
2 3 4
```


### C 作答

In [ ]:
%%c_answer Q11
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q11

In [ ]:
# 檢查 Q11（C）：完成上方 C 程式後執行
_run_check("Q11", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q11
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q11

In [ ]:
# 檢查 Q11（C++）：完成上方 C++ 程式後執行
_run_check("Q11", "CPP")


## Q12. p-- 原地減 1

從尾端開始使用 pointer decrement，將每個元素減 1。

**練習重點：** pointer decrement + write

**範例輸入：**

```text
3
1 2 3
```

**範例輸出：**

```text
0 1 2
```


### C 作答

In [ ]:
%%c_answer Q12
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q12

In [ ]:
# 檢查 Q12（C）：完成上方 C 程式後執行
_run_check("Q12", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q12
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q12

In [ ]:
# 檢查 Q12（C++）：完成上方 C++ 程式後執行
_run_check("Q12", "CPP")


## Q13. begin/end 求和

令 `begin=a`、`end=a+n`，使用 `while(begin!=end)` 求和。

**練習重點：** one-past-the-end

**範例輸入：**

```text
4
1 2 3 4
```

**範例輸出：**

```text
10
```


### C 作答

In [ ]:
%%c_answer Q13
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q13

In [ ]:
# 檢查 Q13（C）：完成上方 C 程式後執行
_run_check("Q13", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q13
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q13

In [ ]:
# 檢查 Q13（C++）：完成上方 C++ 程式後執行
_run_check("Q13", "CPP")


## Q14. 兩位置的 pointer distance

讀入 n、陣列、i、j，建立 pi=a+i、pj=a+j，輸出 `pj-pi`。

**練習重點：** pointer subtraction

**範例輸入：**

```text
5
1 2 3 4 5
1 4
```

**範例輸出：**

```text
3
```


### C 作答

In [ ]:
%%c_answer Q14
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q14

In [ ]:
# 檢查 Q14（C）：完成上方 C 程式後執行
_run_check("Q14", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q14
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q14

In [ ]:
# 檢查 Q14（C++）：完成上方 C++ 程式後執行
_run_check("Q14", "CPP")


## Q15. 找第一個 target 的索引

以 moving pointer 找 target，找到時用 `p-a` 算索引；找不到輸出 -1。

**練習重點：** pointer difference as index

**範例輸入：**

```text
5
4 7 2 7 9
7
```

**範例輸出：**

```text
1
```


### C 作答

In [ ]:
%%c_answer Q15
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q15

In [ ]:
# 檢查 Q15（C）：完成上方 C 程式後執行
_run_check("Q15", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q15
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q15

In [ ]:
# 檢查 Q15（C++）：完成上方 C++ 程式後執行
_run_check("Q15", "CPP")


## Q16. 找最後一個 target 的索引

以 pointer 走訪並用 pointer difference 記錄最後位置。

**練習重點：** pointer difference

**範例輸入：**

```text
5
4 7 2 7 9
7
```

**範例輸出：**

```text
3
```


### C 作答

In [ ]:
%%c_answer Q16
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q16

In [ ]:
# 檢查 Q16（C）：完成上方 C 程式後執行
_run_check("Q16", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q16
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q16

In [ ]:
# 檢查 Q16（C++）：完成上方 C++ 程式後執行
_run_check("Q16", "CPP")


## Q17. 最大最小位置距離

找第一個最大值與第一個最小值的指標，輸出兩者索引距離的絕對值。

**練習重點：** pointer positions

**範例輸入：**

```text
5
3 9 2 7 1
```

**範例輸出：**

```text
3
```


### C 作答

In [ ]:
%%c_answer Q17
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q17

In [ ]:
# 檢查 Q17（C）：完成上方 C 程式後執行
_run_check("Q17", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q17
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q17

In [ ]:
# 檢查 Q17（C++）：完成上方 C++ 程式後執行
_run_check("Q17", "CPP")


## Q18. end 不能解參考

以 `end=a+n` 控制迴圈，只在 p<end 時讀值；輸出元素個數。

**練習重點：** 合法 one-past-end 使用

**範例輸入：**

```text
4
1 2 3 4
```

**範例輸出：**

```text
4
```


### C 作答

In [ ]:
%%c_answer Q18
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q18

In [ ]:
# 檢查 Q18（C）：完成上方 C 程式後執行
_run_check("Q18", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q18
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q18

In [ ]:
# 檢查 Q18（C++）：完成上方 C++ 程式後執行
_run_check("Q18", "CPP")


## Q19. 雙 pointer 複製

使用 src、dst 指標每次都 `++`，把 a 複製到 b。

**練習重點：** parallel pointer traversal

**範例輸入：**

```text
3
2 4 6
```

**範例輸出：**

```text
2 4 6
```


### C 作答

In [ ]:
%%c_answer Q19
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q19

In [ ]:
# 檢查 Q19（C）：完成上方 C 程式後執行
_run_check("Q19", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q19
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q19

In [ ]:
# 檢查 Q19（C++）：完成上方 C++ 程式後執行
_run_check("Q19", "CPP")


## Q20. 雙 pointer 相加

讀入兩個同長陣列，以 pa++、pb++ 同步走訪，輸出對應元素和。

**練習重點：** parallel pointer arithmetic

**範例輸入：**

```text
3
1 2 3
4 5 6
```

**範例輸出：**

```text
5 7 9
```


### C 作答

In [ ]:
%%c_answer Q20
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q20

In [ ]:
# 檢查 Q20（C）：完成上方 C 程式後執行
_run_check("Q20", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q20
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q20

In [ ]:
# 檢查 Q20（C++）：完成上方 C++ 程式後執行
_run_check("Q20", "CPP")


---

# Part C：char* Pointer Arithmetic（Q21–Q30）


## Q21. char* 長度

使用 `char *p` 從字串首端移動到 `\0`，輸出 `p-start`。

**練習重點：** char pointer difference

**範例輸入：**

```text
hello
```

**範例輸出：**

```text
5
```


### C 作答

In [ ]:
%%c_answer Q21
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q21

In [ ]:
# 檢查 Q21（C）：完成上方 C 程式後執行
_run_check("Q21", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q21
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q21

In [ ]:
# 檢查 Q21（C++）：完成上方 C++ 程式後執行
_run_check("Q21", "CPP")


## Q22. char* 反向輸出

令 p 指向最後字元，以 `p--` 反向輸出字串。

**練習重點：** char pointer decrement

**範例輸入：**

```text
hello
```

**範例輸出：**

```text
olleh
```


### C 作答

In [ ]:
%%c_answer Q22
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q22

In [ ]:
# 檢查 Q22（C）：完成上方 C 程式後執行
_run_check("Q22", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q22
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q22

In [ ]:
# 檢查 Q22（C++）：完成上方 C++ 程式後執行
_run_check("Q22", "CPP")


## Q23. 母音計數

用 `char*` 與 `p++` 計算母音數。

**練習重點：** char pointer traversal

**範例輸入：**

```text
education
```

**範例輸出：**

```text
5
```


### C 作答

In [ ]:
%%c_answer Q23
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q23

In [ ]:
# 檢查 Q23（C）：完成上方 C 程式後執行
_run_check("Q23", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q23
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q23

In [ ]:
# 檢查 Q23（C++）：完成上方 C++ 程式後執行
_run_check("Q23", "CPP")


## Q24. 兩端指標反轉字串

使用 left/right pointers 原地交換直到相遇。

**練習重點：** two-pointer arithmetic

**範例輸入：**

```text
abcd
```

**範例輸出：**

```text
dcba
```


### C 作答

In [ ]:
%%c_answer Q24
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q24

In [ ]:
# 檢查 Q24（C）：完成上方 C 程式後執行
_run_check("Q24", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q24
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q24

In [ ]:
# 檢查 Q24（C++）：完成上方 C++ 程式後執行
_run_check("Q24", "CPP")


## Q25. 兩端指標回文

使用 left/right pointer 判斷回文。

**練習重點：** two-ended pointer traversal

**範例輸入：**

```text
racecar
```

**範例輸出：**

```text
1
```


### C 作答

In [ ]:
%%c_answer Q25
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q25

In [ ]:
# 檢查 Q25（C）：完成上方 C 程式後執行
_run_check("Q25", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q25
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q25

In [ ]:
# 檢查 Q25（C++）：完成上方 C++ 程式後執行
_run_check("Q25", "CPP")


## Q26. 數字字元計數

用 pointer traversal 計算字串中的 digit 數。

**練習重點：** char pointer + ctype

**範例輸入：**

```text
a1b23
```

**範例輸出：**

```text
3
```


### C 作答

In [ ]:
%%c_answer Q26
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q26

In [ ]:
# 檢查 Q26（C）：完成上方 C 程式後執行
_run_check("Q26", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q26
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q26

In [ ]:
# 檢查 Q26（C++）：完成上方 C++ 程式後執行
_run_check("Q26", "CPP")


## Q27. 數字字元總和

例如 `a1b23` 的數字值總和是 6。使用 `char*` 走訪。

**練習重點：** pointer + char arithmetic

**範例輸入：**

```text
a1b23
```

**範例輸出：**

```text
6
```


### C 作答

In [ ]:
%%c_answer Q27
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q27

In [ ]:
# 檢查 Q27（C）：完成上方 C 程式後執行
_run_check("Q27", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q27
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q27

In [ ]:
# 檢查 Q27（C++）：完成上方 C++ 程式後執行
_run_check("Q27", "CPP")


## Q28. 第一個指定字元索引

使用 pointer 找 c，找到後輸出 `p-start`。

**練習重點：** pointer distance on string

**範例輸入：**

```text
banana n
```

**範例輸出：**

```text
2
```


### C 作答

In [ ]:
%%c_answer Q28
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q28

In [ ]:
# 檢查 Q28（C）：完成上方 C 程式後執行
_run_check("Q28", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q28
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q28

In [ ]:
# 檢查 Q28（C++）：完成上方 C++ 程式後執行
_run_check("Q28", "CPP")


## Q29. 最後指定字元索引

使用 pointer 找最後一個 c 的位置。

**練習重點：** pointer traversal + distance

**範例輸入：**

```text
banana a
```

**範例輸出：**

```text
5
```


### C 作答

In [ ]:
%%c_answer Q29
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q29

In [ ]:
# 檢查 Q29（C）：完成上方 C 程式後執行
_run_check("Q29", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q29
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q29

In [ ]:
# 檢查 Q29（C++）：完成上方 C++ 程式後執行
_run_check("Q29", "CPP")


## Q30. pointer 取代字元

使用 p++ 將所有 old 字元改成 new。

**練習重點：** pointer mutation

**範例輸入：**

```text
banana a o
```

**範例輸出：**

```text
bonono
```


### C 作答

In [ ]:
%%c_answer Q30
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q30

In [ ]:
# 檢查 Q30（C）：完成上方 C 程式後執行
_run_check("Q30", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q30
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q30

In [ ]:
# 檢查 Q30（C++）：完成上方 C++ 程式後執行
_run_check("Q30", "CPP")


---

# Part D：Two-Pointer 與型態步長（Q31–Q40）


## Q31. double pointer traversal 求和

讀入 n 個 double，使用 `double*` 的 `p++` 求和，輸出小數點後 2 位。

**練習重點：** pointer step depends on type

**範例輸入：**

```text
3
1.5 2.0 3.5
```

**範例輸出：**

```text
7.00
```


### C 作答

In [ ]:
%%c_answer Q31
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q31

In [ ]:
# 檢查 Q31（C）：完成上方 C 程式後執行
_run_check("Q31", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q31
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q31

In [ ]:
# 檢查 Q31（C++）：完成上方 C++ 程式後執行
_run_check("Q31", "CPP")


## Q32. double 陣列全部乘 1.5

透過 `double*` 移動並原地乘 1.5，輸出各值到小數點後 2 位。

**練習重點：** typed pointer arithmetic

**範例輸入：**

```text
3
2 4 6
```

**範例輸出：**

```text
3.00 6.00 9.00
```


### C 作答

In [ ]:
%%c_answer Q32
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q32

In [ ]:
# 檢查 Q32（C）：完成上方 C 程式後執行
_run_check("Q32", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q32
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q32

In [ ]:
# 檢查 Q32（C++）：完成上方 C++ 程式後執行
_run_check("Q32", "CPP")


## Q33. double 平均

使用 begin/end pointer range 求平均，輸出 2 位小數。

**練習重點：** pointer range on double

**範例輸入：**

```text
3
1 2 6
```

**範例輸出：**

```text
3.00
```


### C 作答

In [ ]:
%%c_answer Q33
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q33

In [ ]:
# 檢查 Q33（C）：完成上方 C 程式後執行
_run_check("Q33", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q33
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q33

In [ ]:
# 檢查 Q33（C++）：完成上方 C++ 程式後執行
_run_check("Q33", "CPP")


## Q34. 偏移比較

讀入 i、j，使用 `*(a+i)` 與 `*(a+j)`；前者大輸出 `LEFT`，小輸出 `RIGHT`，相等 `EQUAL`。

**練習重點：** pointer offsets

**範例輸入：**

```text
4
5 1 9 3
0 2
```

**範例輸出：**

```text
RIGHT
```


### C 作答

In [ ]:
%%c_answer Q34
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q34

In [ ]:
# 檢查 Q34（C）：完成上方 C 程式後執行
_run_check("Q34", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q34
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q34

In [ ]:
# 檢查 Q34（C++）：完成上方 C++ 程式後執行
_run_check("Q34", "CPP")


## Q35. 距離區間和

讀入 l、r（含頭含尾），用 p=a+l 到 q=a+r 之間的 pointer traversal 求和。

**練習重點：** pointer range boundaries

**範例輸入：**

```text
5
1 2 3 4 5
1 3
```

**範例輸出：**

```text
9
```


### C 作答

In [ ]:
%%c_answer Q35
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q35

In [ ]:
# 檢查 Q35（C）：完成上方 C 程式後執行
_run_check("Q35", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q35
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q35

In [ ]:
# 檢查 Q35（C++）：完成上方 C++ 程式後執行
_run_check("Q35", "CPP")


## Q36. 兩個特殊位置距離

找第一個負數與第一個正數；若任一不存在輸出 -1，否則輸出索引距離絕對值。

**練習重點：** pointer location tracking

**範例輸入：**

```text
5
0 -2 0 7 9
```

**範例輸出：**

```text
2
```


### C 作答

In [ ]:
%%c_answer Q36
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q36

In [ ]:
# 檢查 Q36（C）：完成上方 C 程式後執行
_run_check("Q36", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q36
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q36

In [ ]:
# 檢查 Q36（C++）：完成上方 C++ 程式後執行
_run_check("Q36", "CPP")


## Q37. 排序陣列的雙指標找 target sum

陣列已升冪。用 left/right pointers 判斷是否存在兩數和為 target，存在輸出 1。

**練習重點：** two-pointer technique

**範例輸入：**

```text
5
1 2 4 7 11
9
```

**範例輸出：**

```text
1
```


### C 作答

In [ ]:
%%c_answer Q37
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q37

In [ ]:
# 檢查 Q37（C）：完成上方 C 程式後執行
_run_check("Q37", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q37
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q37

In [ ]:
# 檢查 Q37（C++）：完成上方 C++ 程式後執行
_run_check("Q37", "CPP")


## Q38. 兩端交換反轉

使用 `left++`、`right--` 原地反轉整數陣列。

**練習重點：** two-ended pointer arithmetic

**範例輸入：**

```text
5
1 2 3 4 5
```

**範例輸出：**

```text
5 4 3 2 1
```


### C 作答

In [ ]:
%%c_answer Q38
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q38

In [ ]:
# 檢查 Q38（C）：完成上方 C 程式後執行
_run_check("Q38", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q38
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q38

In [ ]:
# 檢查 Q38（C++）：完成上方 C++ 程式後執行
_run_check("Q38", "CPP")


## Q39. 循環右移一格

保留最後值，利用 pointer 從尾往前移動元素，最後把保存值放到首端。

**練習重點：** backward pointer movement

**範例輸入：**

```text
4
1 2 3 4
```

**範例輸出：**

```text
4 1 2 3
```


### C 作答

In [ ]:
%%c_answer Q39
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q39

In [ ]:
# 檢查 Q39（C）：完成上方 C 程式後執行
_run_check("Q39", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q39
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q39

In [ ]:
# 檢查 Q39（C++）：完成上方 C++ 程式後執行
_run_check("Q39", "CPP")


## Q40. 左右向中央相加

輸出 `a[0]+a[n-1]`、`a[1]+a[n-2]`...；奇數長度中央元素與自己相加。

**練習重點：** two-pointer convergence

**範例輸入：**

```text
5
1 2 3 4 5
```

**範例輸出：**

```text
6 6 6
```


### C 作答

In [ ]:
%%c_answer Q40
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q40

In [ ]:
# 檢查 Q40（C）：完成上方 C 程式後執行
_run_check("Q40", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q40
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q40

In [ ]:
# 檢查 Q40（C++）：完成上方 C++ 程式後執行
_run_check("Q40", "CPP")


---

# Part E：Pointer Range 綜合（Q41–Q50）


## Q41. rangeSum(begin,end)

撰寫函式接收 `[begin,end)` 兩個 `const int*`，回傳總和。

**練習重點：** pointer range function

**範例輸入：**

```text
4
1 2 3 4
```

**範例輸出：**

```text
10
```


### C 作答

In [ ]:
%%c_answer Q41
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q41

In [ ]:
# 檢查 Q41（C）：完成上方 C 程式後執行
_run_check("Q41", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q41
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q41

In [ ]:
# 檢查 Q41（C++）：完成上方 C++ 程式後執行
_run_check("Q41", "CPP")


## Q42. rangeCountPositive

函式接收 `[begin,end)`，計算正數個數。

**練習重點：** generic pointer range

**範例輸入：**

```text
5
1 -2 3 0 4
```

**範例輸出：**

```text
3
```


### C 作答

In [ ]:
%%c_answer Q42
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q42

In [ ]:
# 檢查 Q42（C）：完成上方 C 程式後執行
_run_check("Q42", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q42
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q42

In [ ]:
# 檢查 Q42（C++）：完成上方 C++ 程式後執行
_run_check("Q42", "CPP")


## Q43. copyRange

用兩個 source pointers 與 destination pointer 複製整個範圍。

**練習重點：** range copy

**範例輸入：**

```text
3
1 9 4
```

**範例輸出：**

```text
1 9 4
```


### C 作答

In [ ]:
%%c_answer Q43
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q43

In [ ]:
# 檢查 Q43（C）：完成上方 C 程式後執行
_run_check("Q43", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q43
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q43

In [ ]:
# 檢查 Q43（C++）：完成上方 C++ 程式後執行
_run_check("Q43", "CPP")


## Q44. equalRange

讀入兩個同長陣列，用 pointer ranges 判斷是否完全相同。

**練習重點：** parallel ranges

**範例輸入：**

```text
3
1 2 3
1 2 3
```

**範例輸出：**

```text
1
```


### C 作答

In [ ]:
%%c_answer Q44
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q44

In [ ]:
# 檢查 Q44（C）：完成上方 C 程式後執行
_run_check("Q44", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q44
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q44

In [ ]:
# 檢查 Q44（C++）：完成上方 C++ 程式後執行
_run_check("Q44", "CPP")


## Q45. lexicographicCompare

讀入兩個同長整數陣列。用 pointers 找第一個不同元素；a<b 輸出 -1、a>b 輸出 1、完全相同輸出 0。

**練習重點：** pointer parallel comparison

**範例輸入：**

```text
3
1 2 3
1 4 0
```

**範例輸出：**

```text
-1
```


### C 作答

In [ ]:
%%c_answer Q45
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q45

In [ ]:
# 檢查 Q45（C）：完成上方 C 程式後執行
_run_check("Q45", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q45
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q45

In [ ]:
# 檢查 Q45（C++）：完成上方 C++ 程式後執行
_run_check("Q45", "CPP")


## Q46. read/write pointers 移除零

原地把所有非零元素往前壓縮，輸出新長度與壓縮後元素；若長度 0，第二行輸出 EMPTY。

**練習重點：** read/write pointer pattern

**範例輸入：**

```text
6
1 0 2 0 0 3
```

**範例輸出：**

```text
3
1 2 3
```


### C 作答

In [ ]:
%%c_answer Q46
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q46

In [ ]:
# 檢查 Q46（C）：完成上方 C 程式後執行
_run_check("Q46", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q46
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q46

In [ ]:
# 檢查 Q46（C++）：完成上方 C++ 程式後執行
_run_check("Q46", "CPP")


## Q47. 移除相鄰重複

陣列已排序。使用 read/write pointers 原地保留每個不同值一次，輸出新長度與內容。

**練習重點：** two moving pointers

**範例輸入：**

```text
7
1 1 2 2 2 5 5
```

**範例輸出：**

```text
3
1 2 5
```


### C 作答

In [ ]:
%%c_answer Q47
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q47

In [ ]:
# 檢查 Q47（C）：完成上方 C 程式後執行
_run_check("Q47", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q47
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q47

In [ ]:
# 檢查 Q47（C++）：完成上方 C++ 程式後執行
_run_check("Q47", "CPP")


## Q48. 合併兩個排序陣列

用 pa、pb 與 output pointer 合併兩個升冪陣列。

**練習重點：** multiple pointer traversal

**範例輸入：**

```text
3 3
1 4 7
2 3 8
```

**範例輸出：**

```text
1 2 3 4 7 8
```


### C 作答

In [ ]:
%%c_answer Q48
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q48

In [ ]:
# 檢查 Q48（C）：完成上方 C 程式後執行
_run_check("Q48", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q48
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q48

In [ ]:
# 檢查 Q48（C++）：完成上方 C++ 程式後執行
_run_check("Q48", "CPP")


## Q49. 兩排序陣列共同值數量

用 two pointers 計算兩個排序陣列中共同元素的配對數；相同值每次匹配後兩邊都前進。

**練習重點：** two-array two-pointer

**範例輸入：**

```text
4 5
1 2 4 7
2 3 4 4 8
```

**範例輸出：**

```text
2
```


### C 作答

In [ ]:
%%c_answer Q49
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q49

In [ ]:
# 檢查 Q49（C）：完成上方 C 程式後執行
_run_check("Q49", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q49
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q49

In [ ]:
# 檢查 Q49（C++）：完成上方 C++ 程式後執行
_run_check("Q49", "CPP")


## Q50. Pointer Arithmetic 總整合

升冪陣列中找第一組和為 target 的兩數；用 left/right pointers。找到輸出兩個索引，找不到輸出 `-1 -1`。

**練習重點：** two pointers + pointer difference

**範例輸入：**

```text
5
1 2 4 7 11
9
```

**範例輸出：**

```text
1 3
```


### C 作答

In [ ]:
%%c_answer Q50
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q50

In [ ]:
# 檢查 Q50（C）：完成上方 C 程式後執行
_run_check("Q50", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q50
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q50

In [ ]:
# 檢查 Q50（C++）：完成上方 C++ 程式後執行
_run_check("Q50", "CPP")


---

# 總檢查


In [ ]:
_final_report()
